<a href="https://colab.research.google.com/github/BilalAsifB/Agentic-AI-ASG1/blob/main/solution/question_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Completed pseudo-code:

``` python
function multi_head_attention(Q, K, V, num_heads, d_model, mask=None):
    batch_size = Q.shape[0]
    d_k = d_model / num_heads

    Q_proj = Q @ W_Q  # (batch, seq_len, d_model)
    K_proj = K @ W_K  # (batch, seq_len, d_model)
    V_proj = V @ W_V  # (batch, seq_len, d_model)

    Q_heads = split_heads(Q_proj, num_heads)  # (batch, num_heads, seq_len, d_k)
    K_heads = split_heads(K_proj, num_heads)
    V_heads = split_heads(V_proj, num_heads)   

    attention_output, attention_weights = scaled_dot_product_attention(
        Q_heads, K_heads, V_heads, mask
    )

    attention_output = concatenate_heads(attention_output, num_heads)  
    output = attention_output @ W_O  # (batch, seq_len, d_model)

    return output, attention_weights
end function
```

### Learning agent:

In [2]:
# Imports

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [3]:
# Building blocks

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q, K, V : (batch, num_heads, seq_len, d_k)
    Returns  : output (same shape), weights (batch, num_heads, seq_len, seq_len)
    """
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, V)
    return output, attention_weights


def split_heads(x, num_heads):
    """
    (batch, seq_len, d_model)  →  (batch, num_heads, seq_len, d_k)
    """
    batch_size, seq_len, d_model = x.shape
    d_k = d_model // num_heads
    x = x.view(batch_size, seq_len, num_heads, d_k)
    return x.transpose(1, 2)


def concatenate_heads(x, num_heads):
    """
    (batch, num_heads, seq_len, d_k)  →  (batch, seq_len, d_model)
    """
    batch_size, _, seq_len, d_k = x.shape
    x = x.transpose(1, 2).contiguous()
    return x.view(batch_size, seq_len, num_heads * d_k)

In [4]:
# Multi-head attention

def multi_head_attention(Q, K, V, W_Q, W_K, W_V, W_O,
                         num_heads, d_model, mask=None):
    """
    Q, K, V  : (batch, seq_len, d_model)
    W_Q/K/V  : (d_model, d_model)
    W_O      : (d_model, d_model)
    """
    # linear projections
    Q_proj = torch.matmul(Q, W_Q)   # (batch, seq_len, d_model)
    K_proj = torch.matmul(K, W_K)   # (batch, seq_len, d_model)
    V_proj = torch.matmul(V, W_V)   # (batch, seq_len, d_model)

    # split into heads
    Q_heads = split_heads(Q_proj, num_heads)   # (batch, num_heads, seq_len, d_k)
    K_heads = split_heads(K_proj, num_heads)
    V_heads = split_heads(V_proj, num_heads)

    # scaled dot-product attention
    attn_out, attn_weights = scaled_dot_product_attention(
        Q_heads, K_heads, V_heads, mask
    )

    # merge heads
    attn_out = concatenate_heads(attn_out, num_heads)  # (batch, seq_len, d_model)

    # final projection
    output = torch.matmul(attn_out, W_O)               # (batch, seq_len, d_model)
    return output, attn_weights

class MultiHeadAttention(nn.Module):
    """Trainable multi-head attention with learnable W_Q, W_K, W_V, W_O."""

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model    = d_model
        self.num_heads  = num_heads
        self.d_k        = d_model // num_heads

        # Learnable projections
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)
        self._reset_parameters()

    def _reset_parameters(self):
        for w in [self.W_Q, self.W_K, self.W_V, self.W_O]:
            nn.init.xavier_uniform_(w.weight)

    def forward(self, Q, K, V, mask=None):
        """
        Q, K, V : (batch, seq_len, d_model)
        mask    : (batch, 1, seq_len, seq_len) or None
        """
        # Linear projections + split heads
        Q_heads = split_heads(self.W_Q(Q), self.num_heads)
        K_heads = split_heads(self.W_K(K), self.num_heads)
        V_heads = split_heads(self.W_V(V), self.num_heads)

        # Attention
        attn_out, attn_weights = scaled_dot_product_attention(
            Q_heads, K_heads, V_heads, mask
        )
        attn_out = self.dropout(attn_out)

        # Merge + project
        output = self.W_O(concatenate_heads(attn_out, self.num_heads))
        return output, attn_weights


In [5]:
# Learning-agent extension

class TransformerBlock(nn.Module):
    """
    One encoder block used by the learning agent.
    Applies MHA → Add & Norm → FFN → Add & Norm.
    """

    def __init__(self, d_model: int, num_heads: int,
                 d_ff: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.attention  = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1      = nn.LayerNorm(d_model)
        self.norm2      = nn.LayerNorm(d_model)
        self.ffn        = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.dropout    = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-attention + residual
        attn_out, weights = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))

        # Feed-forward + residual
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x, weights


class LearningAgent(nn.Module):
    """
    A simple transformer-based learning agent.
    Stacks N encoder blocks, then produces a policy logits vector.

    Architecture
    ────────────
    Input tokens
        → Embedding + positional encoding
        → N × TransformerBlock  (self-attention + FFN)
        → mean-pool over sequence
        → policy head (linear → action logits)
    """

    def __init__(self, vocab_size: int, d_model: int, num_heads: int,
                 num_layers: int, num_actions: int,
                 max_seq_len: int = 512, d_ff: int = 2048,
                 dropout: float = 0.1):
        super().__init__()

        self.embedding  = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc    = self._build_pos_encoding(max_seq_len, d_model)
        self.dropout    = nn.Dropout(dropout)

        self.blocks     = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # Policy head
        self.policy_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, num_actions),
        )
        # Value head (for actor-critic / PPO)
        self.value_head  = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, 1),
        )

    @staticmethod
    def _build_pos_encoding(max_len: int, d_model: int) -> torch.Tensor:
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe.unsqueeze(0)          # (1, max_len, d_model)

    def _causal_mask(self, seq_len: int, device) -> torch.Tensor:
        """Upper-triangular mask so each position only attends to past tokens."""
        return torch.tril(torch.ones(seq_len, seq_len, device=device)).unsqueeze(0).unsqueeze(0)

    def forward(self, token_ids: torch.Tensor, use_causal_mask: bool = True):
        """
        token_ids : (batch, seq_len)  — integer token IDs
        Returns   : action_logits (batch, num_actions),
                    state_value   (batch, 1),
                    all_weights   list of (batch, heads, seq, seq)
        """
        B, S = token_ids.shape
        device = token_ids.device

        # Embed + positional encoding
        x = self.embedding(token_ids)
        x = self.dropout(x + self.pos_enc[:, :S, :].to(device))

        mask = self._causal_mask(S, device) if use_causal_mask else None

        all_weights = []
        for block in self.blocks:
            x, w = block(x, mask)
            all_weights.append(w)

        # Mean-pool across sequence → single context vector
        context = x.mean(dim=1)

        action_logits = self.policy_head(context)
        state_value   = self.value_head(context)

        return action_logits, state_value, all_weights

In [10]:
def perform_test():
    """
    Tests the LearningAgent class with dummy inputs and prints the shapes
    of the action logits, state value, and attention weights to verify
    that the forward pass is working correctly.
    """
    BATCH, SEQ, D_MODEL, HEADS = 2, 10, 64, 8
    VOCAB, ACTIONS, LAYERS     = 1000, 5, 3

    agent = LearningAgent(
        vocab_size=VOCAB, d_model=D_MODEL, num_heads=HEADS,
        num_layers=LAYERS, num_actions=ACTIONS
    )

    tokens = torch.randint(1, VOCAB, (BATCH, SEQ))
    logits, value, weights = agent(tokens)

    print(f"Action logits : {logits.shape}")
    print(f"State value   : {value.shape}")
    print(f"Attn weights  : {weights[0].shape}")

In [11]:
if __name__ == "__main__":
    perform_test()

Action logits : torch.Size([2, 5])
State value   : torch.Size([2, 1])
Attn weights  : torch.Size([2, 8, 10, 10])
